<a href="https://colab.research.google.com/github/Naveen-gale/DATA-SCENS-LEARNING/blob/main/novelai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate sentencepiece

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
BASE_PATH = "/content/drive/MyDrive/AntigoneGPT"

MODEL_SAVE_PATH = f"{BASE_PATH}/final_model"

DATASET_PATH = f"{BASE_PATH}/final_dataset.jsonl"

### Reloading and Reprocessing Data

Since you've added new data, we need to reload and reprocess the dataset to incorporate the changes. This ensures our model trains on the most current information.

In [34]:
import json

# Initialize an empty list to store the data
data = []

# Open the dataset file and load each JSON line
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

# Print the total number of samples and the first sample to verify
print(f"Total number of samples: {len(data)}")
print(f"First sample: {data[0]}")

Total number of samples: 1549
First sample: {'instruction': 'What is the character of Polyneices like in the passage?', 'response': 'Polyneices is portrayed as a commander who rouses his troops with windy phrases, suggesting he may be more focused on rhetoric than actual leadership, and is likened to a wild eagle, implying a fierce and potentially threatening presence.'}


In [35]:
# Prepare the data in the required "User: Question\n\nAssistant: Answer" format
texts = []

for row in data:
    sample = f"""User: {row['instruction']}

Assistant: {row['response']}"""
    texts.append(sample)

# Print the first formatted text to verify
print("First formatted text sample:")
print(texts[0])

First formatted text sample:
User: What is the character of Polyneices like in the passage?

Assistant: Polyneices is portrayed as a commander who rouses his troops with windy phrases, suggesting he may be more focused on rhetoric than actual leadership, and is likened to a wild eagle, implying a fierce and potentially threatening presence.


In [36]:
from datasets import Dataset

# Create a Hugging Face Dataset from the prepared texts
dataset = Dataset.from_dict({"text": texts})

# Display the dataset information
display(dataset)

Dataset({
    features: ['text'],
    num_rows: 1549
})

In [37]:
import math

def tokenize(example):
    # Tokenize the input text, truncating to max_length and padding to max_length
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

# Apply the tokenization function to the entire dataset
tokenized = dataset.map(tokenize, batched=True)

# Display the tokenized dataset information
display(tokenized)

# Calculate and print the approximate number of training steps for warmup_steps calculation
train_samples = math.ceil(len(tokenized) * (1 - 0.05)) # Assuming 0.05 test_size
approx_total_steps = (5 * train_samples) / 2 # epochs * (train_samples / batch_size)
print(f"Approximate total training steps: {approx_total_steps}")

Map:   0%|          | 0/1549 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1549
})

Approximate total training steps: 3680.0


In [38]:
# Split the tokenized dataset into training and evaluation sets
split = tokenized.train_test_split(test_size=0.05, seed=42) # Added seed for reproducibility

train_dataset = split["train"]

eval_dataset = split["test"]

# Print information about the training and evaluation datasets
print("Training Dataset:")
print(train_dataset)

print("Evaluation Dataset:")
print(eval_dataset)

Training Dataset:
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1471
})
Evaluation Dataset:
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 78
})


In [20]:
import json

data = []

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print(len(data))
print(data[0])

1549
{'instruction': 'What is the character of Polyneices like in the passage?', 'response': 'Polyneices is portrayed as a commander who rouses his troops with windy phrases, suggesting he may be more focused on rhetoric than actual leadership, and is likened to a wild eagle, implying a fierce and potentially threatening presence.'}


In [22]:
texts = []

for row in data:

    sample = f"""User: {row['instruction']}

Assistant: {row['response']}"""

    texts.append(sample)

print(texts[0])

User: What is the character of Polyneices like in the passage?

Assistant: Polyneices is portrayed as a commander who rouses his troops with windy phrases, suggesting he may be more focused on rhetoric than actual leadership, and is likened to a wild eagle, implying a fierce and potentially threatening presence.


In [23]:
from datasets import Dataset

dataset = Dataset.from_dict({"text": texts})
dataset

Dataset({
    features: ['text'],
    num_rows: 1549
})

In [24]:
from transformers import GPT2TokenizerFast

In [25]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [26]:
def tokenize(example):

    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized = dataset.map(tokenize, batched=True)

tokenized

Map:   0%|          | 0/1549 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1549
})

In [27]:
split = tokenized.train_test_split(test_size=0.05)

train_dataset = split["train"]

eval_dataset = split["test"]

print(train_dataset)

print(eval_dataset)

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1471
})
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 78
})


In [28]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2")

model.resize_token_embeddings(len(tokenizer))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Embedding(50257, 768)

In [39]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(

    output_dir="./checkpoints",

    num_train_epochs=5, # Reduced from 10 to 5 to combat overfitting

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    learning_rate=3e-5, # Slightly reduced learning rate

    weight_decay=0.05, # Increased weight decay for regularization

    warmup_steps=184, # Calculated as 5% of approx_total_steps (3680 * 0.05)

    logging_steps=25,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    fp16=True,

    report_to="none"
)

In [40]:
from transformers import Trainer, DataCollatorForLanguageModeling, EarlyStoppingCallback

data_collator = DataCollatorForLanguageModeling(

    tokenizer=tokenizer,

    mlm=False
)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    data_collator=data_collator,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Add EarlyStoppingCallback
)

In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.104540,1.459279
2,0.932236,1.446834
3,0.839649,1.461764
4,0.824751,1.485265
5,0.882015,1.479336


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=3680, training_loss=0.9073043212942455, metrics={'train_runtime': 806.5338, 'train_samples_per_second': 9.119, 'train_steps_per_second': 4.563, 'total_flos': 1921802895360000.0, 'train_loss': 0.9073043212942455, 'epoch': 5.0})

In [42]:
trainer.save_model(MODEL_SAVE_PATH)

tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!


In [43]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MODEL_SAVE_PATH,
    tokenizer=MODEL_SAVE_PATH
)

prompt = """User: Who is Antigone?

Assistant:"""

result = pipe(
    prompt,
    max_new_tokens=150,
    do_sample=False,
    temperature=0.3
)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: Who is Antigone?

Assistant: Antigone is a prominent figure in the scene, serving as a catalyst for the events that unfold. Her character is closely tied to the themes of love, loyalty, and morality, and her actions are driven by a sense of duty and duty to her family. Antigone's character is also closely tied to the idea of justice, as she is willing to stand up for what she believes in, even if it means going against the will of others. Her actions are ultimately motivated by a sense of duty and duty to her family, as she is willing to risk her own life for what she believes in. Antigone's character is also closely tied to the idea of morality, as she is willing to stand up for what she believes in
